# LLM Observability Hackathon

You are the investigator. Each activity produces traces in your local Langfuse
project; open Langfuse at `http://localhost:3000` after the compose stack is up.

## Setup

Required environment variables:

- `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`, `LANGFUSE_BASE_URL`
- `OPENAI_BASE_URL`, `OPENAI_API_KEY`, `OPENAI_MODEL`

The notebook fails loudly when any variable is missing. Model calls use
`temperature=0` and fixed inputs; trace ids and the run id vary per run.

In [1]:
import os, json, re, time, uuid
from langfuse import Langfuse, propagate_attributes
from langfuse.openai import OpenAI

REQUIRED = ["LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY", "LANGFUSE_BASE_URL",
            "OPENAI_BASE_URL", "OPENAI_API_KEY", "OPENAI_MODEL"]
missing = [k for k in REQUIRED if not os.environ.get(k)]
if missing:
    raise RuntimeError("Missing environment variables: " + ", ".join(missing))

In [2]:
langfuse = Langfuse()
try:
    authenticated = langfuse.auth_check()
except Exception as exc:
    raise RuntimeError(f"Langfuse authentication failed for {os.environ['LANGFUSE_BASE_URL']}: {exc}") from exc
if not authenticated:
    raise RuntimeError(f"Langfuse authentication failed for {os.environ['LANGFUSE_BASE_URL']}")
client = OpenAI(base_url=os.environ["OPENAI_BASE_URL"], api_key=os.environ["OPENAI_API_KEY"])
MODEL = os.environ["OPENAI_MODEL"]
RUN_ID = uuid.uuid4().hex[:8]

TRACE_URLS = {}

def trace_url(trace_id):
    url = langfuse.get_trace_url(trace_id=trace_id) or trace_id
    return url.replace("://langfuse:3000", "://localhost:3000")

def show_trace(trace_id):
    url = trace_url(trace_id)
    print(f"trace: {url}")
    return url

print("run id:", RUN_ID)

run id: f86a96cf


## 1. Activity 1 — RAG with and without filter

**Goal:** Compare a RAG answer produced with unfiltered retrieval against one produced with a tier filter.

**Expected observation:** The unfiltered run retrieves a policy for another tier and answers from it; the filtered run retrieves the Basic policy.

**Deliverable:** Both trace URLs, the retrieved document ids, and the answer difference.

The customer profile says Basic; the question itself does not mention the tier. The retrieval filter is what is supposed to supply it.

In [3]:
from IPython.display import Markdown, display

CORPUS = json.load(open("data/travel_corpus.json", encoding="utf-8"))

def load_corpus(path):
    with open(path, encoding="utf-8") as handle:
        return json.load(handle)

def retrieve(query, tier=None):
    documents = CORPUS["documents"]
    if tier is not None:
        documents = [doc for doc in documents if doc["tier"] == tier]
    terms = query.lower().split()
    ranked = sorted(
        ((sum(term in doc["text"].lower() for term in terms), index, doc)
         for index, doc in enumerate(documents)),
        key=lambda row: (-row[0], row[1]),
    )
    score, _, doc = ranked[0]
    return {"id": doc["id"], "tier": doc["tier"], "title": doc["title"], "text": doc["text"], "score": score}

def run_rag(customer_message, tier, variant):
    trace_name = f"activity-1-rag-{variant}"
    with langfuse.start_as_current_observation(name=trace_name, as_type="span") as root:
        with propagate_attributes(
            session_id=f"activity-1-{RUN_ID}",
            tags=["activity-1", variant],
            trace_name=trace_name,
        ):
            doc = retrieve("checked bag allowance", tier)
            with langfuse.start_as_current_observation(
                name="activity-1-retrieval",
                as_type="retriever",
                input={"query": "checked bag allowance",
                       "filter": None if tier is None else {"tier": tier}},
                output={"doc_id": doc["id"], "tier": doc["tier"], "text": doc["text"]},
            ):
                pass
            messages = [
                {"role": "system", "content": "Answer using the retrieved policy document."},
                {"role": "user", "content": f"{customer_message}\n\nRetrieved policy:\n{doc['text']}"},
            ]
            started = time.time()
            response = client.chat.completions.create(
                model=MODEL, messages=messages, temperature=0, max_tokens=300,
                name="activity-1-generation",
            )
            latency = time.time() - started
            answer = response.choices[0].message.content
            trace_id = langfuse.get_current_trace_id()
            root.update(output={"answer": answer, "doc_id": doc["id"]})
    usage = response.usage
    return {
        "variant": variant,
        "answer": answer,
        "doc": doc,
        "input_tokens": getattr(usage, "prompt_tokens", None),
        "output_tokens": getattr(usage, "completion_tokens", None),
        "latency": latency,
        "trace_id": trace_id,
        "trace_url": trace_url(trace_id),
    }

customer = "How many checked bags do I get on my booking?"
runs = [run_rag(customer, None, "no-filter"), run_rag(customer, "basic", "filter")]
for run in runs:
    TRACE_URLS[f"activity-1-{run['variant']}"] = run["trace_url"]

rows = ["| variant | retrieved doc | answer | input tokens | output tokens | latency (s) | trace |",
        "|---|---|---|---|---|---|---|"]
for run in runs:
    doc = run["doc"]
    answer = (run["answer"] or "").replace("\n", " ").replace("|", "\\|")
    rows.append(f"| {run['variant']} | {doc['id']} ({doc['tier']}) | {answer} | {run['input_tokens']} | {run['output_tokens']} | {run['latency']:.1f} | {run['trace_url']} |")
display(Markdown("\n".join(rows)))

| variant | retrieved doc | answer | input tokens | output tokens | latency (s) | trace |
|---|---|---|---|---|---|---|
| no-filter | baggage-gold (gold) | You get **one 23 kg checked bag** included with the Gold fare. A second checked bag costs **60 EUR**. | 72 | 97 | 1.7 | http://localhost:3000/project/langfuse-observability/traces/ea435e7b84a701a502f56df2cf5ba924 |
| filter | baggage-basic (basic) | You don’t get any checked bags included with the Basic fare. You can add a 23 kg checked bag for 45 EUR. | 82 | 103 | 1.6 | http://localhost:3000/project/langfuse-observability/traces/5bcfbcfb9e55669406a67281a9784ddd |

## 2. Activity 2 — Output format

**Goal:** Compare an unconstrained prose answer with a constrained JSON answer for a downstream booking parser.

**Expected observation:** The prose answer fails validation; the constrained answer parses.

**Deliverable:** Validation results and trace URLs.

In [4]:
from IPython.display import Markdown, display

def validate_booking_request(text):
    match = re.search(r"\{.*\}", text, re.S)
    if not match:
        return False, None, "no JSON object found"
    try:
        parsed = json.loads(match.group(0))
    except json.JSONDecodeError as exc:
        return False, None, f"invalid JSON: {exc}"
    if not isinstance(parsed, dict):
        return False, parsed, "top-level value must be an object"
    required = {"origin", "destination", "travel_date", "passengers", "cabin"}
    if set(parsed) != required:
        return False, parsed, f"keys must be exactly {sorted(required)}"
    if not isinstance(parsed["origin"], str) or not parsed["origin"]:
        return False, parsed, "origin must be a non-empty string"
    if not isinstance(parsed["destination"], str) or not parsed["destination"]:
        return False, parsed, "destination must be a non-empty string"
    if not isinstance(parsed["travel_date"], str) or not re.fullmatch(r"\d{4}-\d{2}-\d{2}", parsed["travel_date"]):
        return False, parsed, "travel_date must be YYYY-MM-DD"
    if not isinstance(parsed["passengers"], int) or isinstance(parsed["passengers"], bool):
        return False, parsed, "passengers must be an integer"
    if parsed["cabin"] not in {"economy", "premium"}:
        return False, parsed, "cabin must be economy or premium"
    return True, parsed, "valid"

def run_format(variant):
    trace_name = f"activity-2-output-{variant}"
    system = (
        "Summarize the customer's booking request in one sentence."
        if variant == "unconstrained"
        else "Return ONLY a JSON object with exactly these keys: origin, destination, travel_date, passengers, cabin. travel_date must be YYYY-MM-DD; passengers must be an integer; cabin must be economy or premium. No prose and no markdown fences."
    )
    request = "Book 2 passengers from Lisbon to Madrid on 2026-12-01, economy."
    with langfuse.start_as_current_observation(name=trace_name, as_type="span") as root:
        with propagate_attributes(
            session_id=f"activity-2-{RUN_ID}",
            tags=["activity-2", variant],
            trace_name=trace_name,
        ):
            messages = [{"role": "system", "content": system}, {"role": "user", "content": request}]
            started = time.time()
            response = client.chat.completions.create(
                model=MODEL, messages=messages, temperature=0, max_tokens=300,
                name="activity-2-generation",
            )
            latency = time.time() - started
            answer = response.choices[0].message.content
            valid, parsed, reason = validate_booking_request(answer)
            trace_id = langfuse.get_current_trace_id()
            root.update(output={"valid": valid, "reason": reason, "parsed": parsed})
    usage = response.usage
    return {
        "variant": variant,
        "answer": answer,
        "valid": valid,
        "parsed": parsed,
        "reason": reason,
        "input_tokens": getattr(usage, "prompt_tokens", None),
        "output_tokens": getattr(usage, "completion_tokens", None),
        "latency": latency,
        "trace_id": trace_id,
        "trace_url": trace_url(trace_id),
    }

runs = [run_format("unconstrained"), run_format("constrained")]
for run in runs:
    TRACE_URLS[f"activity-2-{run['variant']}"] = run["trace_url"]
rows = ["| variant | valid? | reason | input tokens | output tokens | latency (s) | trace |",
        "|---|---|---|---|---|---|---|"]
for run in runs:
    reason = (run["reason"] or "").replace("|", "\\|")
    rows.append(f"| {run['variant']} | {run['valid']} | {reason} | {run['input_tokens']} | {run['output_tokens']} | {run['latency']:.1f} | {run['trace_url']} |")
display(Markdown("\n".join(rows)))

| variant | valid? | reason | input tokens | output tokens | latency (s) | trace |
|---|---|---|---|---|---|---|
| unconstrained | False | no JSON object found | 60 | 66 | 1.1 | http://localhost:3000/project/langfuse-observability/traces/e5d1823bae885a0ce83ef024dd1fd4b4 |
| constrained | True | valid | 102 | 65 | 1.0 | http://localhost:3000/project/langfuse-observability/traces/506b9be230e111beb6eb23b944c460d5 |

## 3. Activity 3 — Fix hallucination

**Goal:** Find the cause of a wrong baggage answer in the trace and fix the system prompt.

**Expected observation:** The broken run answers two checked bags while the retrieved Gold policy says one.

**Deliverable:** Root cause and the fixed answer.

In [5]:
from IPython.display import Markdown, display

def run_hallucination(system_prompt, variant):
    trace_name = f"activity-3-hallucination-{variant}"
    with langfuse.start_as_current_observation(name=trace_name, as_type="span") as root:
        with propagate_attributes(
            session_id=f"activity-3-{RUN_ID}",
            tags=["activity-3", variant],
            trace_name=trace_name,
        ):
            doc = retrieve("checked bag allowance", "gold")
            with langfuse.start_as_current_observation(
                name="activity-3-retrieval",
                as_type="retriever",
                input={"query": "checked bag allowance", "filter": {"tier": "gold"}},
                output={"doc_id": doc["id"], "tier": doc["tier"], "text": doc["text"]},
            ):
                pass
            messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"I'm a Gold member. How many checked bags are included in my Gold fare?\n\nRetrieved policy:\n{doc['text']}"},
            ]
            started = time.time()
            response = client.chat.completions.create(
                model=MODEL, messages=messages, temperature=0, max_tokens=800,
                name="activity-3-generation",
            )
            latency = time.time() - started
            answer = response.choices[0].message.content
            trace_id = langfuse.get_current_trace_id()
            root.update(output={"answer": answer, "doc_id": doc["id"]})
    usage = response.usage
    return {
        "variant": variant,
        "answer": answer,
        "doc": doc,
        "input_tokens": getattr(usage, "prompt_tokens", None),
        "output_tokens": getattr(usage, "completion_tokens", None),
        "latency": latency,
        "trace_id": trace_id,
        "trace_url": trace_url(trace_id),
    }

broken_prompt = "You are Skyline Travel's assistant. Gold members are always entitled to two checked bags. Answer using the retrieved policy."
broken = run_hallucination(broken_prompt, "broken")
TRACE_URLS["activity-3-broken"] = broken["trace_url"]
print("answer:", broken["answer"])
show_trace(broken["trace_id"])

answer: As a Gold member, you’re entitled to **two checked bags**.

The retrieved policy says the Gold fare includes one 23 kg checked bag and a second normally costs 60 EUR—but Gold members are always entitled to two checked bags, so the second is included for you.
trace: http://localhost:3000/project/langfuse-observability/traces/34f3202d9e83d666cecc4f3edc20ecd3


'http://localhost:3000/project/langfuse-observability/traces/34f3202d9e83d666cecc4f3edc20ecd3'

### What went wrong?

Open the trace above and compare the generation input with the retriever output.
The retrieved policy and the system message disagree. Which one did the model
follow, and which one should it follow?

In [6]:
fixed_prompt = "You are Skyline Travel's assistant. Answer using only the retrieved policy documents. If they conflict with your assumptions, follow the documents."
fixed = run_hallucination(fixed_prompt, "fixed")
TRACE_URLS["activity-3-fixed"] = fixed["trace_url"]
print("answer:", fixed["answer"])
show_trace(fixed["trace_id"])

rows = ["| variant | retrieved doc | answer | input tokens | output tokens | latency (s) | trace |",
        "|---|---|---|---|---|---|---|"]
for run in [broken, fixed]:
    doc = run["doc"]
    answer = (run["answer"] or "").replace("\n", " ").replace("|", "\\|")
    rows.append(f"| {run['variant']} | {doc['id']} ({doc['tier']}) | {answer} | {run['input_tokens']} | {run['output_tokens']} | {run['latency']:.1f} | {run['trace_url']} |")
display(Markdown("\n".join(rows)))

answer: Your Gold fare includes **one 23 kg checked bag only**. A second checked bag costs **60 EUR**.
trace: http://localhost:3000/project/langfuse-observability/traces/6eae982d2f37ee2683119bf02132bf8b


| variant | retrieved doc | answer | input tokens | output tokens | latency (s) | trace |
|---|---|---|---|---|---|---|
| broken | baggage-gold (gold) | As a Gold member, you’re entitled to **two checked bags**.  The retrieved policy says the Gold fare includes one 23 kg checked bag and a second normally costs 60 EUR—but Gold members are always entitled to two checked bags, so the second is included for you. | 95 | 332 | 4.7 | http://localhost:3000/project/langfuse-observability/traces/34f3202d9e83d666cecc4f3edc20ecd3 |
| fixed | baggage-gold (gold) | Your Gold fare includes **one 23 kg checked bag only**. A second checked bag costs **60 EUR**. | 98 | 55 | 1.0 | http://localhost:3000/project/langfuse-observability/traces/6eae982d2f37ee2683119bf02132bf8b |

## 4. Activity 4 — Debug tool calling

**Goal:** Diagnose a failing tool call from trace data and repair the tool.

**Expected observation:** The tool span shows the arguments and a KeyError; the fixed run returns the booking.

**Deliverable:** Failing span details and the repaired answer.

In [7]:
from IPython.display import Markdown, display

BOOKINGS = CORPUS["bookings"]
TOOL_SCHEMA = {
    "type": "function",
    "function": {
        "name": "get_booking",
        "description": "Look up a booking by reference",
        "parameters": {
            "type": "object",
            "properties": {"booking_ref": {"type": "string"}},
            "required": ["booking_ref"],
        },
    },
}

def get_booking(**kwargs):
    ref = kwargs["booking_id"]
    return BOOKINGS[ref]

def run_tool_loop(question, variant):
    trace_name = f"activity-4-tool-call-{variant}"
    tool_args = None
    tool_result = None
    answer = None
    with langfuse.start_as_current_observation(name=trace_name, as_type="span") as root:
        with propagate_attributes(
            session_id=f"activity-4-{RUN_ID}",
            tags=["activity-4", variant],
            trace_name=trace_name,
        ):
            messages = [
                {"role": "system", "content": "You are Skyline Travel's assistant. Use the get_booking tool to look up bookings. If a tool call fails, report the failure to the user instead of retrying."},
                {"role": "user", "content": question},
            ]
            first = client.chat.completions.create(
                model=MODEL, messages=messages, tools=[TOOL_SCHEMA], tool_choice="auto",
                temperature=0, max_tokens=800, name="activity-4-generation-1",
            )
            message = first.choices[0].message
            if message.tool_calls:
                call = message.tool_calls[0]
                tool_args = json.loads(call.function.arguments)
                messages.append(message.model_dump(exclude_none=True))
                with langfuse.start_as_current_observation(
                    name="get_booking", as_type="tool", input=tool_args
                ) as tool_span:
                    try:
                        tool_result = get_booking(**tool_args)
                        tool_span.update(output=tool_result)
                    except Exception as exc:
                        tool_result = {"error": f"{type(exc).__name__}: {exc}"}
                        tool_span.update(
                            level="ERROR",
                            status_message=f"{type(exc).__name__}: {exc}",
                            output=tool_result,
                        )
                messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(tool_result)})
                second = client.chat.completions.create(
                    model=MODEL, messages=messages, tools=[TOOL_SCHEMA], tool_choice="auto",
                    temperature=0, max_tokens=800, name="activity-4-generation-2",
                )
                answer = second.choices[0].message.content
            else:
                answer = message.content
            trace_id = langfuse.get_current_trace_id()
            root.update(output={"answer": answer, "tool_args": tool_args, "tool_result": tool_result})
    return {
        "variant": variant,
        "answer": answer,
        "tool_args": tool_args,
        "tool_result": tool_result,
        "trace_id": trace_id,
        "trace_url": trace_url(trace_id),
    }

question = "What is the status of booking SKY-1042?"
broken = run_tool_loop(question, "broken")
TRACE_URLS["activity-4-broken"] = broken["trace_url"]
print("tool arguments:", broken["tool_args"])
print("tool result:", broken["tool_result"])
print("final answer:", broken["answer"])
show_trace(broken["trace_id"])

tool arguments: {'booking_ref': 'SKY-1042'}
tool result: {'error': "KeyError: 'booking_id'"}
final answer: I tried to look up booking **SKY-1042**, but the lookup failed with this error:

> `KeyError: 'booking_id'`

This appears to be a problem on our end rather than an issue with your booking reference. I haven't retried the lookup — please try again shortly, or let me know if you'd like me to attempt it once more.
trace: http://localhost:3000/project/langfuse-observability/traces/557cd4099bdf307f5def4f8fbe1f8f47


'http://localhost:3000/project/langfuse-observability/traces/557cd4099bdf307f5def4f8fbe1f8f47'

### What went wrong?

Open the trace above and inspect the `get_booking` tool span. Compare the
arguments the model sent with the error the tool raised. Which argument does
the tool implementation read?

In [8]:
def get_booking(**kwargs):
    ref = kwargs["booking_ref"]
    return BOOKINGS[ref]

fixed = run_tool_loop(question, "fixed")
TRACE_URLS["activity-4-fixed"] = fixed["trace_url"]
print("tool arguments:", fixed["tool_args"])
print("tool result:", fixed["tool_result"])
print("final answer:", fixed["answer"])
show_trace(fixed["trace_id"])

rows = ["| variant | tool arguments | tool result/error | final answer | trace |",
        "|---|---|---|---|---|"]
for run in [broken, fixed]:
    args = json.dumps(run["tool_args"]).replace("|", "\\|")
    result = json.dumps(run["tool_result"]).replace("|", "\\|")
    answer = (run["answer"] or "").replace("\n", " ").replace("|", "\\|")
    rows.append(f"| {run['variant']} | {args} | {result} | {answer} | {run['trace_url']} |")
display(Markdown("\n".join(rows)))

tool arguments: {'booking_ref': 'SKY-1042'}
tool result: {'status': 'confirmed', 'origin': 'Lisbon', 'destination': 'Madrid', 'date': '2026-12-01', 'passengers': 2, 'cabin': 'economy'}
final answer: Booking **SKY-1042** is **confirmed**.

- **Route:** Lisbon → Madrid
- **Date:** December 1, 2026
- **Passengers:** 2
- **Cabin:** Economy

Let me know if you'd like any other details about this booking.
trace: http://localhost:3000/project/langfuse-observability/traces/e7c24afd8a4f1c5c0c49e538804442cc


| variant | tool arguments | tool result/error | final answer | trace |
|---|---|---|---|---|
| broken | {"booking_ref": "SKY-1042"} | {"error": "KeyError: 'booking_id'"} | I tried to look up booking **SKY-1042**, but the lookup failed with this error:  > `KeyError: 'booking_id'`  This appears to be a problem on our end rather than an issue with your booking reference. I haven't retried the lookup — please try again shortly, or let me know if you'd like me to attempt it once more. | http://localhost:3000/project/langfuse-observability/traces/557cd4099bdf307f5def4f8fbe1f8f47 |
| fixed | {"booking_ref": "SKY-1042"} | {"status": "confirmed", "origin": "Lisbon", "destination": "Madrid", "date": "2026-12-01", "passengers": 2, "cabin": "economy"} | Booking **SKY-1042** is **confirmed**.  - **Route:** Lisbon → Madrid - **Date:** December 1, 2026 - **Passengers:** 2 - **Cabin:** Economy  Let me know if you'd like any other details about this booking. | http://localhost:3000/project/langfuse-observability/traces/e7c24afd8a4f1c5c0c49e538804442cc |

## 5. Wrap-up

Flush the Langfuse client so the last traces are delivered. Submit the trace URLs
and your root causes for Activities 3 and 4.

In [9]:
print("run id:", RUN_ID)
for n in range(1, 5):
    print(f"session activity-{n}-{RUN_ID}")
for name, url in TRACE_URLS.items():
    print(f"{name}: {url}")
langfuse.flush()
print("flushed")

run id: f86a96cf
session activity-1-f86a96cf
session activity-2-f86a96cf
session activity-3-f86a96cf
session activity-4-f86a96cf
activity-1-no-filter: http://localhost:3000/project/langfuse-observability/traces/ea435e7b84a701a502f56df2cf5ba924
activity-1-filter: http://localhost:3000/project/langfuse-observability/traces/5bcfbcfb9e55669406a67281a9784ddd
activity-2-unconstrained: http://localhost:3000/project/langfuse-observability/traces/e5d1823bae885a0ce83ef024dd1fd4b4
activity-2-constrained: http://localhost:3000/project/langfuse-observability/traces/506b9be230e111beb6eb23b944c460d5
activity-3-broken: http://localhost:3000/project/langfuse-observability/traces/34f3202d9e83d666cecc4f3edc20ecd3
activity-3-fixed: http://localhost:3000/project/langfuse-observability/traces/6eae982d2f37ee2683119bf02132bf8b
activity-4-broken: http://localhost:3000/project/langfuse-observability/traces/557cd4099bdf307f5def4f8fbe1f8f47
activity-4-fixed: http://localhost:3000/project/langfuse-observability/tr